# Self-attention, geometrically
Two-word sentences with 2-number embeddings, so that every vector can be drawn. All numbers are hypothetical (chosen by hand, not trained).

In [1]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

E = {"money": np.array([2.0, 7.0]), "bank": np.array([7.0, 3.0]), "river": np.array([8.0, -2.0])}
W_Q = np.array([[0.2, 0.1], [0.1, 0.3]])
W_K = np.array([[0.3, 0.0], [0.1, 0.2]])
W_V = np.array([[0.9, 0.2], [0.1, 0.8]])

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def self_attention(words):
    X = np.stack([E[w] for w in words])          # one row per word
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    S = Q @ K.T                                  # similarity scores
    A = softmax(S / np.sqrt(2))                  # d_k = 2
    return X, Q, K, V, S, A, A @ V

## "money bank": every step

In [2]:
rows = []
for words in (["money", "bank"], ["river", "bank"]):
    X, Q, K, V, S, A, Y = self_attention(words)
    print(" ".join(words))
    for i, w in enumerate(words):
        print(f"  {w:6s} e {X[i]}  q {Q[i]}  k {K[i]}  v {V[i]}")
    print("  scores\n", S, "\n  scaled\n", S / np.sqrt(2), "\n  weights\n", A, "\n  outputs y\n", Y)
    for i, w in enumerate(words):
        for kind, M in (("e", X), ("q", Q), ("k", K), ("v", V), ("y", Y)):
            rows.append(dict(sentence=" ".join(words), word=w, kind=kind, x=M[i, 0], y=M[i, 1]))
        for j, w2 in enumerate(words):
            rows.append(dict(sentence=" ".join(words), word=w, kind=f"weight_{w2}", x=A[i, j], y=S[i, j]))
pd.DataFrame(rows).to_csv("data/vectors.csv", index=False, float_format="%.4f")

money bank
  money  e [2. 7.]  q [1.1 2.3]  k [1.3 1.4]  v [2.5 6. ]
  bank   e [7. 3.]  q [1.7 1.6]  k [2.4 0.6]  v [6.6 3.8]
  scores
 [[4.65 4.02]
 [4.45 5.04]] 
  scaled
 [[3.288 2.843]
 [3.147 3.564]] 
  weights
 [[0.61  0.39 ]
 [0.397 0.603]] 
  outputs y
 [[4.101 5.141]
 [4.972 4.674]]
river bank
  river  e [ 8. -2.]  q [1.4 0.2]  k [ 2.2 -0.4]  v [7. 0.]
  bank   e [7. 3.]  q [1.7 1.6]  k [2.4 0.6]  v [6.6 3.8]
  scores
 [[3.   3.48]
 [3.1  5.04]] 
  scaled
 [[2.121 2.461]
 [2.192 3.564]] 
  weights
 [[0.416 0.584]
 [0.202 0.798]] 
  outputs y
 [[6.766 2.219]
 [6.681 3.031]]


## The output lies on the segment between the two value vectors
The weights are positive and sum to 1, so y_bank = v_bank + w_money (v_money - v_bank): a point on the segment from v_bank to v_money, a fraction w_money of the way.

In [3]:
for words in (["money", "bank"], ["river", "bank"]):
    X, Q, K, V, S, A, Y = self_attention(words)
    w_other = A[1, 0]
    assert np.allclose(A.sum(1), 1)
    assert np.allclose(Y[1], V[1] + w_other * (V[0] - V[1]))
    moved = np.linalg.norm(Y[1] - V[1])
    print(f"{words[0]:6s} bank: y_bank = {Y[1]}, {w_other:.3f} of the way from v_bank to v_{words[0]}, "
          f"moved {moved:.2f} units")
print("y_bank in the two sentences differs by", np.linalg.norm(self_attention(["money", "bank"])[6][1]
                                                - self_attention(["river", "bank"])[6][1]).round(2))

money  bank: y_bank = [4.972 4.674], 0.397 of the way from v_bank to v_money, moved 1.85 units
river  bank: y_bank = [6.681 3.031], 0.202 of the way from v_bank to v_river, moved 0.77 units
y_bank in the two sentences differs by 2.37


## The output as a change added to the word's vector
In a transformer the attention output is added back to the word's own vector (the residual connection of the
transformer encoder Note): new vector = e + Δe, with Δe the attention output. Here W_O is the identity, so Δe = y.

In [2]:
for words in (["money", "bank"], ["river", "bank"]):
    X, Q, K, V, S, A, Y = self_attention(words)
    e, de = X[1], Y[1]
    new = e + de
    ang = lambda v: np.degrees(np.arctan2(v[1], v[0]))
    print(f"{' '.join(words):10s}: e_bank {e}, Δe {de.round(2)}, e + Δe {new.round(2)}; "
          f"angle {ang(e):.1f} -> {ang(new):.1f} degrees; |Δe|/|e| = {np.linalg.norm(de) / np.linalg.norm(e):.2f}")

money bank: e_bank [7. 3.], Δe [4.97 4.67], e + Δe [11.97  7.67]; angle 23.2 -> 32.7 degrees; |Δe|/|e| = 0.90
river bank: e_bank [7. 3.], Δe [6.68 3.03], e + Δe [13.68  6.03]; angle 23.2 -> 23.8 degrees; |Δe|/|e| = 0.96


## A trained model: how big is the change? (GPT-2 small)
GPT-2 small (Radford et al. 2019; 12 blocks, 768 numbers per token) adds each attention block's output to every
token's vector, exactly as above. We run the published weights in NumPy and measure, for the last token "bank",
|Δe| / |e| and the cosine between the vector before and after each attention block.
The weights (523 MB) and the tokenizer are downloaded from Hugging Face on the first run (needs internet) and cached.

In [3]:
import json, struct
from pathlib import Path
import keras
from tokenizers import Tokenizer

URL = "https://huggingface.co/openai-community/gpt2/resolve/main/"
path = keras.utils.get_file("gpt2_model.safetensors", URL + "model.safetensors")
tok = Tokenizer.from_file(keras.utils.get_file("gpt2_tokenizer.json", URL + "tokenizer.json"))
raw = open(path, "rb").read()
hlen = struct.unpack("<Q", raw[:8])[0]
header = json.loads(raw[8:8 + hlen])
G = {k: np.frombuffer(raw[8 + hlen + v["data_offsets"][0]:8 + hlen + v["data_offsets"][1]], "<f4").reshape(v["shape"])
     for k, v in header.items() if k != "__metadata__"}


def ln(x, g, b):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + 1e-5) * g + b


def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))


def gpt2(ids):
    """GPT-2 small forward pass (pre-norm blocks, causal mask). Returns the logits and, per block, the vectors
    entering the attention block and the attention block's output (the change it adds)."""
    n = len(ids)
    x = G["wte.weight"][ids] + G["wpe.weight"][:n]
    before, change = [], []
    for l in range(12):
        p = f"h.{l}."
        a = ln(x, G[p + "ln_1.weight"], G[p + "ln_1.bias"]) @ G[p + "attn.c_attn.weight"] + G[p + "attn.c_attn.bias"]
        q, k, v = (t.reshape(n, 12, 64).transpose(1, 0, 2) for t in np.split(a, 3, -1))
        S = q @ k.transpose(0, 2, 1) / 8 + np.triu(np.full((n, n), -np.inf), 1)
        A = np.exp(S - S.max(-1, keepdims=True))
        A /= A.sum(-1, keepdims=True)
        de = (A @ v).transpose(1, 0, 2).reshape(n, 768) @ G[p + "attn.c_proj.weight"] + G[p + "attn.c_proj.bias"]
        before.append(x.copy())
        change.append(de)
        x = x + de                                                     # the residual connection: e + Δe
        h = gelu(ln(x, G[p + "ln_2.weight"], G[p + "ln_2.bias"]) @ G[p + "mlp.c_fc.weight"] + G[p + "mlp.c_fc.bias"])
        x = x + h @ G[p + "mlp.c_proj.weight"] + G[p + "mlp.c_proj.bias"]
    return ln(x, G["ln_f.weight"], G["ln_f.bias"]) @ G["wte.weight"].T, before, change


ids = tok.encode("Michael Jordan plays the sport of").ids
top = gpt2(ids)[0][-1].argsort()[::-1][:3]
print("check: 'Michael Jordan plays the sport of' ->", [tok.decode([t]) for t in top])
rows = []
for s in ("He sat on the river bank", "She works at the money bank"):
    ids = tok.encode(s).ids
    _, before, change = gpt2(ids)
    for l in range(12):
        e, de = before[l][-1], change[l][-1]
        rows.append(dict(sentence=s, block=l + 1, ratio=np.linalg.norm(de) / np.linalg.norm(e),
                         cosine=e @ (e + de) / np.linalg.norm(e) / np.linalg.norm(e + de)))
delta = pd.DataFrame(rows)
print(delta.pivot(index="block", columns="sentence", values=["ratio", "cosine"]).round(2))
mid = delta[delta.block.between(2, 11)]
print("blocks 2-11: |Δe|/|e| from", mid.ratio.min().round(2), "to", mid.ratio.max().round(2),
      "; mean", mid.ratio.mean().round(2), "; cosine at least", mid.cosine.min().round(3))
delta.round(4).to_csv("data/gpt2_delta.csv", index=False)

check: 'Michael Jordan plays the sport of' -> [' basketball', ' football', ' golf']


                            ratio                              \
sentence He sat on the river bank She works at the money bank   
block                                                           
1                            6.41                        5.92   
2                            0.25                        0.19   
3                            0.15                        0.15   
4                            0.16                        0.13   
5                            0.09                        0.14   
6                            0.14                        0.13   
7                            0.14                        0.14   
8                            0.21                        0.14   
9                            0.13                        0.12   
10                           0.20                        0.13   
11                           0.24                        0.23   
12                           1.10                        1.16   

                        